In [ ]:
# ============================================================
# Agentic Cyber-SLA Intelligence for Secure UAV-6G Aerial CPS
# Kaggle T4 GPU Ready Code
# ============================================================
# Author: Nizamuddin Maitlo
# Theme: Agentic AI + Cybersecurity + SLA Prediction + UAV + 6G + Aerial CPS
# ============================================================

import os
import gc
import glob
import json
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. Reproducibility and GPU setup
# ============================================================

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

OUT_DIR = "/kaggle/working/Agentic_Cyber_SLA_Results"
os.makedirs(OUT_DIR, exist_ok=True)

FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
MODEL_DIR = os.path.join(OUT_DIR, "models")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ============================================================
# 2. Auto-detect datasets from Kaggle input
# ============================================================

all_csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("CSV files found:")
for f in all_csv_files:
    print(" -", f)

def read_csv_safe(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, encoding="latin1")

sla_path = None
hetero_path = None
cyber_path = None

for f in all_csv_files:
    name = os.path.basename(f).lower()
    if "telemetry" in name or "6g_fl_telemetry" in name:
        sla_path = f
    elif "heterogeneity" in name or "6g_fl_heterogeneity" in name:
        hetero_path = f
    elif "domain" in name or "malicious" in name or "dns" in name:
        cyber_path = f

print("\nDetected paths:")
print("SLA telemetry:", sla_path)
print("SLA heterogeneity:", hetero_path)
print("Cyber dataset:", cyber_path)

if sla_path is None:
    raise FileNotFoundError("SLA telemetry CSV not found. Add your SLA dataset to Kaggle input.")

if cyber_path is None:
    print("\nWARNING: Cyber dataset not auto-detected.")
    print("Please make sure the Kaggle malicious-domain dataset is added to this notebook.")
    print("The code will still run after you manually set cyber_path.")
    raise FileNotFoundError("Cyber dataset CSV not found.")

sla_df = read_csv_safe(sla_path)
cyber_df = read_csv_safe(cyber_path)

print("\nSLA shape:", sla_df.shape)
print("Cyber shape:", cyber_df.shape)

print("\nSLA columns:")
print(sla_df.columns.tolist())

print("\nCyber columns:")
print(cyber_df.columns.tolist())

# ============================================================
# 3. Helper functions
# ============================================================

def find_target_column(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

def reduce_memory(df):
    for col in df.columns:
        if df[col].dtype == "float64":
            df[col] = df[col].astype("float32")
        elif df[col].dtype == "int64":
            if df[col].min() >= np.iinfo(np.int32).min and df[col].max() <= np.iinfo(np.int32).max:
                df[col] = df[col].astype("int32")
    return df

def clean_binary_target(y):
    if y.dtype == "object":
        y = y.astype(str).str.lower().str.strip()
        mapping = {
            "malicious": 1, "benign": 0,
            "bad": 1, "good": 0,
            "1": 1, "0": 0,
            "true": 1, "false": 0,
            "yes": 1, "no": 0
        }
        y = y.map(mapping)
    return y.astype(int)

def prepare_tabular_features(df, target_col, drop_cols=None, max_cat_cardinality=50):
    df = df.copy()
    if drop_cols is None:
        drop_cols = []
    
    drop_cols = [c for c in drop_cols if c in df.columns]
    y = clean_binary_target(df[target_col])
    
    X = df.drop(columns=[target_col] + drop_cols)
    
    # Remove columns with too many missing values
    missing_ratio = X.isna().mean()
    X = X.loc[:, missing_ratio < 0.75]
    
    # Remove constant columns
    nunique = X.nunique(dropna=True)
    X = X.loc[:, nunique > 1]
    
    cat_cols = []
    num_cols = []
    
    for c in X.columns:
        if X[c].dtype == "object" or str(X[c].dtype).startswith("category"):
            if X[c].nunique(dropna=True) <= max_cat_cardinality:
                cat_cols.append(c)
            else:
                # For high-cardinality text columns such as domain names, create numeric text features
                s = X[c].astype(str)
                X[c + "_len"] = s.str.len()
                X[c + "_digits"] = s.str.count(r"\d")
                X[c + "_dots"] = s.str.count(r"\.")
                X[c + "_hyphens"] = s.str.count(r"\-")
                X[c + "_vowels"] = s.str.count(r"[aeiouAEIOU]")
                X = X.drop(columns=[c])
        else:
            num_cols.append(c)
    
    # Recalculate after high-cardinality expansion
    cat_cols = []
    num_cols = []
    for c in X.columns:
        if X[c].dtype == "object" or str(X[c].dtype).startswith("category"):
            if X[c].nunique(dropna=True) <= max_cat_cardinality:
                cat_cols.append(c)
        else:
            num_cols.append(c)
    
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
        X[c] = X[c].fillna(X[c].median())
    
    for c in cat_cols:
        X[c] = X[c].astype(str).fillna("missing")
    
    if len(cat_cols) > 0:
        X = pd.get_dummies(X, columns=cat_cols, dummy_na=False)
    
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)
    
    return X.astype("float32"), y.astype("int32")

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    
    metrics = {}
    metrics["Accuracy"] = accuracy_score(y_true, y_pred)
    metrics["Precision"] = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"] = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1"] = f1_score(y_true, y_pred, zero_division=0)
    
    try:
        metrics["ROC_AUC"] = roc_auc_score(y_true, y_prob)
    except Exception:
        metrics["ROC_AUC"] = np.nan
    
    try:
        metrics["PR_AUC"] = average_precision_score(y_true, y_prob)
    except Exception:
        metrics["PR_AUC"] = np.nan
    
    return metrics

def save_metrics_table(metrics_dict, filename):
    df = pd.DataFrame(metrics_dict).T
    df = df.round(4)
    path = os.path.join(TAB_DIR, filename)
    df.to_csv(path)
    print("Saved:", path)
    return df

# ============================================================
# 4. PyTorch Dataset and Model
# ============================================================

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class AgenticTabularNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim // 2, 1)
        )
    
    def forward(self, x):
        return self.net(x).squeeze(1)

def train_torch_model(
    X_train, y_train, X_val, y_val,
    model_name="agent",
    epochs=25,
    batch_size=1024,
    lr=1e-3,
    weight_decay=1e-4,
    patience=6
):
    train_ds = TabularDataset(X_train, y_train)
    val_ds = TabularDataset(X_val, y_val)
    
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )
    
    model = AgenticTabularNet(input_dim=X_train.shape[1]).to(DEVICE)
    
    pos_weight_value = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    
    best_auc = -1
    best_state = None
    bad_epochs = 0
    history = []
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0
        
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(xb)
                loss = criterion(logits, yb)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * len(yb)
        
        scheduler.step()
        train_loss /= len(train_ds)
        
        model.eval()
        val_probs = []
        val_targets = []
        
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                logits = model(xb)
                probs = torch.sigmoid(logits).detach().cpu().numpy()
                val_probs.extend(probs)
                val_targets.extend(yb.numpy())
        
        val_probs = np.array(val_probs)
        val_targets = np.array(val_targets)
        val_metrics = compute_metrics(val_targets, val_probs)
        val_auc = val_metrics["ROC_AUC"]
        
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics
        })
        
        print(
            f"[{model_name}] Epoch {epoch:02d} | "
            f"Loss {train_loss:.4f} | "
            f"F1 {val_metrics['F1']:.4f} | "
            f"AUC {val_metrics['ROC_AUC']:.4f} | "
            f"PR-AUC {val_metrics['PR_AUC']:.4f}"
        )
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = model.state_dict()
            bad_epochs = 0
        else:
            bad_epochs += 1
        
        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch}")
            break
    
    model.load_state_dict(best_state)
    
    model_path = os.path.join(MODEL_DIR, f"{model_name}.pt")
    torch.save(model.state_dict(), model_path)
    
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(TAB_DIR, f"{model_name}_training_history.csv"), index=False)
    
    return model, hist_df

def predict_torch_model(model, X, batch_size=2048):
    ds = torch.tensor(X, dtype=torch.float32)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    
    model.eval()
    probs = []
    
    with torch.no_grad():
        for xb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            prob = torch.sigmoid(logits).detach().cpu().numpy()
            probs.extend(prob)
    
    return np.array(probs)

# ============================================================
# 5. Prepare Cyber Risk Agent data
# ============================================================

cyber_df = reduce_memory(cyber_df)

cyber_target_candidates = [
    "label", "Label", "class", "Class", "target", "Target",
    "is_malicious", "malicious", "Malicious"
]

cyber_target = find_target_column(cyber_df, cyber_target_candidates)

if cyber_target is None:
    raise ValueError(
        "Cyber target column not found. Rename your target column to label, class, target, or is_malicious."
    )

print("\nCyber target column:", cyber_target)

cyber_drop_cols = []
for c in cyber_df.columns:
    cl = c.lower()
    if cl in ["id", "index"]:
        cyber_drop_cols.append(c)

X_cyber, y_cyber = prepare_tabular_features(
    cyber_df,
    target_col=cyber_target,
    drop_cols=cyber_drop_cols,
    max_cat_cardinality=30
)

print("Prepared cyber X:", X_cyber.shape)
print("Cyber label distribution:")
print(pd.Series(y_cyber).value_counts(normalize=True))

Xc_train, Xc_temp, yc_train, yc_temp = train_test_split(
    X_cyber, y_cyber,
    test_size=0.30,
    random_state=SEED,
    stratify=y_cyber
)

Xc_val, Xc_test, yc_val, yc_test = train_test_split(
    Xc_temp, yc_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=yc_temp
)

cyber_scaler = StandardScaler()
Xc_train_s = cyber_scaler.fit_transform(Xc_train).astype("float32")
Xc_val_s = cyber_scaler.transform(Xc_val).astype("float32")
Xc_test_s = cyber_scaler.transform(Xc_test).astype("float32")

print("\nCyber train/val/test:", Xc_train_s.shape, Xc_val_s.shape, Xc_test_s.shape)

# ============================================================
# 6. Train Cyber Risk Agent
# ============================================================

cyber_agent, cyber_history = train_torch_model(
    Xc_train_s, yc_train.values,
    Xc_val_s, yc_val.values,
    model_name="Cyber_Risk_Agent",
    epochs=30,
    batch_size=2048,
    lr=1e-3,
    patience=7
)

cyber_test_prob = predict_torch_model(cyber_agent, Xc_test_s)
cyber_metrics = compute_metrics(yc_test.values, cyber_test_prob)

print("\nCyber Risk Agent Test Metrics:")
print(cyber_metrics)

# ============================================================
# 7. Prepare SLA Reliability Agent data
# ============================================================

sla_df = reduce_memory(sla_df)

sla_target = find_target_column(sla_df, ["sla_ok_next", "SLA_OK_NEXT", "target", "label"])

if sla_target is None:
    raise ValueError("SLA target column not found. Expected sla_ok_next.")

print("\nSLA target column:", sla_target)

sla_drop_cols = []
for c in ["timestamp_s"]:
    if c in sla_df.columns:
        sla_drop_cols.append(c)

# Group-aware split by client_id to avoid leakage
if "client_id" in sla_df.columns:
    groups = sla_df["client_id"]
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_idx, temp_idx = next(splitter.split(sla_df, sla_df[sla_target], groups))
    
    sla_train_df = sla_df.iloc[train_idx].reset_index(drop=True)
    sla_temp_df = sla_df.iloc[temp_idx].reset_index(drop=True)
    
    groups_temp = sla_temp_df["client_id"]
    splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_idx, test_idx = next(splitter2.split(sla_temp_df, sla_temp_df[sla_target], groups_temp))
    
    sla_val_df = sla_temp_df.iloc[val_idx].reset_index(drop=True)
    sla_test_df = sla_temp_df.iloc[test_idx].reset_index(drop=True)
else:
    sla_train_df, sla_temp_df = train_test_split(
        sla_df, test_size=0.30, random_state=SEED, stratify=sla_df[sla_target]
    )
    sla_val_df, sla_test_df = train_test_split(
        sla_temp_df, test_size=0.50, random_state=SEED, stratify=sla_temp_df[sla_target]
    )

X_sla_train, y_sla_train = prepare_tabular_features(
    sla_train_df,
    target_col=sla_target,
    drop_cols=sla_drop_cols,
    max_cat_cardinality=50
)

X_sla_val, y_sla_val = prepare_tabular_features(
    sla_val_df,
    target_col=sla_target,
    drop_cols=sla_drop_cols,
    max_cat_cardinality=50
)

X_sla_test, y_sla_test = prepare_tabular_features(
    sla_test_df,
    target_col=sla_target,
    drop_cols=sla_drop_cols,
    max_cat_cardinality=50
)

# Align columns
X_sla_val = X_sla_val.reindex(columns=X_sla_train.columns, fill_value=0)
X_sla_test = X_sla_test.reindex(columns=X_sla_train.columns, fill_value=0)

sla_scaler = StandardScaler()
Xs_train_s = sla_scaler.fit_transform(X_sla_train).astype("float32")
Xs_val_s = sla_scaler.transform(X_sla_val).astype("float32")
Xs_test_s = sla_scaler.transform(X_sla_test).astype("float32")

print("\nPrepared SLA X:", Xs_train_s.shape, Xs_val_s.shape, Xs_test_s.shape)
print("SLA label distribution:")
print(pd.Series(y_sla_train).value_counts(normalize=True))

# ============================================================
# 8. Train SLA Reliability Agent
# ============================================================

sla_agent, sla_history = train_torch_model(
    Xs_train_s, y_sla_train.values,
    Xs_val_s, y_sla_val.values,
    model_name="SLA_Reliability_Agent",
    epochs=35,
    batch_size=1024,
    lr=1e-3,
    patience=8
)

sla_test_prob = predict_torch_model(sla_agent, Xs_test_s)
sla_metrics = compute_metrics(y_sla_test.values, sla_test_prob)

print("\nSLA Reliability Agent Test Metrics:")
print(sla_metrics)

# ============================================================
# 9. Agentic Cyber-SLA Decision Fusion
# ============================================================

# Cyber risk = P(malicious)
cyber_risk = cyber_test_prob

# SLA failure risk = 1 - P(sla_ok_next)
sla_failure_risk = 1.0 - sla_test_prob

# Decision-level fusion.
# These datasets are not sample-aligned; therefore, fusion is performed at the decision-framework level.
# We pair test probabilities by truncating to the minimum test length.
n_fusion = min(len(cyber_risk), len(sla_failure_risk))

cyber_risk_f = cyber_risk[:n_fusion]
sla_failure_risk_f = sla_failure_risk[:n_fusion]

alpha = 0.55
beta = 0.45

cyber_sla_risk = alpha * cyber_risk_f + beta * sla_failure_risk_f

def agentic_action(cyber_r, sla_r):
    if cyber_r < 0.40 and sla_r < 0.40:
        return "EDGE_OFFLOAD"
    elif cyber_r < 0.40 and sla_r >= 0.40:
        return "LOCAL_OR_DELAY"
    elif cyber_r >= 0.40 and sla_r < 0.40:
        return "BLOCK_ALERT"
    else:
        return "SAFE_MODE_ISOLATE"

actions = [agentic_action(c, s) for c, s in zip(cyber_risk_f, sla_failure_risk_f)]

fusion_df = pd.DataFrame({
    "cyber_risk_prob": cyber_risk_f,
    "sla_failure_risk_prob": sla_failure_risk_f,
    "cyber_sla_risk_score": cyber_sla_risk,
    "agentic_action": actions
})

fusion_df.to_csv(os.path.join(TAB_DIR, "agentic_cyber_sla_decisions.csv"), index=False)

action_summary = fusion_df["agentic_action"].value_counts().reset_index()
action_summary.columns = ["Agentic_Action", "Count"]
action_summary["Percentage"] = 100 * action_summary["Count"] / action_summary["Count"].sum()
action_summary.to_csv(os.path.join(TAB_DIR, "agentic_action_summary.csv"), index=False)

print("\nAgentic Action Summary:")
print(action_summary)

# ============================================================
# 10. Baseline comparison
# ============================================================

# Define unsafe condition:
# unsafe = malicious or SLA failure.
# Since datasets are decision-level paired, this is used for framework evaluation.
y_cyber_true_f = yc_test.values[:n_fusion]
y_sla_true_f = y_sla_test.values[:n_fusion]

unsafe_true = np.logical_or(y_cyber_true_f == 1, y_sla_true_f == 0).astype(int)

cyber_only_score = cyber_risk_f
sla_only_score = sla_failure_risk_f
joint_score = cyber_sla_risk

baseline_metrics = {
    "Cyber_Only_Agent": compute_metrics(unsafe_true, cyber_only_score),
    "SLA_Only_Agent": compute_metrics(unsafe_true, sla_only_score),
    "Agentic_Cyber_SLA_Agent": compute_metrics(unsafe_true, joint_score)
}

baseline_df = save_metrics_table(baseline_metrics, "cvpr_style_baseline_comparison.csv")
print("\nCVPR-style Baseline Comparison:")
print(baseline_df)

# ============================================================
# 11. Regime-wise SLA evaluation
# ============================================================

if "regime" in sla_test_df.columns:
    regime_results = []
    temp = sla_test_df.copy()
    temp["sla_prob"] = sla_test_prob
    temp["sla_pred"] = (temp["sla_prob"] >= 0.5).astype(int)
    
    for regime, g in temp.groupby("regime"):
        yy = g[sla_target].astype(int).values
        pp = g["sla_prob"].values
        mm = compute_metrics(yy, pp)
        mm["Regime"] = regime
        mm["N"] = len(g)
        regime_results.append(mm)
    
    regime_df = pd.DataFrame(regime_results)
    regime_df = regime_df[["Regime", "N", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]]
    regime_df = regime_df.round(4)
    regime_df.to_csv(os.path.join(TAB_DIR, "regime_wise_sla_results.csv"), index=False)
    
    print("\nRegime-wise SLA Results:")
    print(regime_df)

# ============================================================
# 12. Publication-quality plots
# ============================================================

def plot_roc(y_true, y_prob, title, filename):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate", fontsize=12)
    plt.ylabel("True Positive Rate", fontsize=12)
    plt.title(title, fontsize=13)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, filename)
    plt.savefig(path, dpi=600, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def plot_pr(y_true, y_prob, title, filename):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    
    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"AP = {ap:.4f}")
    plt.xlabel("Recall", fontsize=12)
    plt.ylabel("Precision", fontsize=12)
    plt.title(title, fontsize=13)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, filename)
    plt.savefig(path, dpi=600, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def plot_confusion(y_true, y_prob, title, filename):
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title, fontsize=13)
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("True", fontsize=12)
    plt.xticks([0, 1], ["0", "1"])
    plt.yticks([0, 1], ["0", "1"])
    
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=12)
    
    plt.colorbar()
    plt.tight_layout()
    path = os.path.join(FIG_DIR, filename)
    plt.savefig(path, dpi=600, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def plot_bar_action_summary(action_summary):
    plt.figure(figsize=(7, 4))
    plt.bar(action_summary["Agentic_Action"], action_summary["Percentage"])
    plt.ylabel("Percentage (%)", fontsize=12)
    plt.xlabel("Agentic Decision", fontsize=12)
    plt.title("Agentic Cyber-SLA Decision Distribution", fontsize=13)
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, "agentic_action_distribution_600dpi.png")
    plt.savefig(path, dpi=600, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

plot_roc(yc_test.values, cyber_test_prob, "Cyber Risk Agent ROC Curve", "cyber_agent_roc_600dpi.png")
plot_pr(yc_test.values, cyber_test_prob, "Cyber Risk Agent PR Curve", "cyber_agent_pr_600dpi.png")
plot_confusion(yc_test.values, cyber_test_prob, "Cyber Risk Agent Confusion Matrix", "cyber_agent_confusion_600dpi.png")

plot_roc(y_sla_test.values, sla_test_prob, "SLA Reliability Agent ROC Curve", "sla_agent_roc_600dpi.png")
plot_pr(y_sla_test.values, sla_test_prob, "SLA Reliability Agent PR Curve", "sla_agent_pr_600dpi.png")
plot_confusion(y_sla_test.values, sla_test_prob, "SLA Reliability Agent Confusion Matrix", "sla_agent_confusion_600dpi.png")

plot_roc(unsafe_true, joint_score, "Agentic Cyber-SLA Risk ROC Curve", "agentic_cyber_sla_roc_600dpi.png")
plot_pr(unsafe_true, joint_score, "Agentic Cyber-SLA Risk PR Curve", "agentic_cyber_sla_pr_600dpi.png")
plot_confusion(unsafe_true, joint_score, "Agentic Cyber-SLA Risk Confusion Matrix", "agentic_cyber_sla_confusion_600dpi.png")

plot_bar_action_summary(action_summary)

# ============================================================
# 13. Save final combined result summary
# ============================================================

final_metrics = {
    "Cyber_Risk_Agent": cyber_metrics,
    "SLA_Reliability_Agent": sla_metrics,
    "Agentic_Cyber_SLA_Agent": compute_metrics(unsafe_true, joint_score)
}

final_df = save_metrics_table(final_metrics, "final_cvpr_results_summary.csv")
print("\nFinal CVPR-style Results Summary:")
print(final_df)

# ============================================================
# 14. Generate LaTeX tables for paper
# ============================================================

latex_baseline = baseline_df.to_latex(
    index=True,
    float_format="%.4f",
    caption="Baseline comparison of cyber-only, SLA-only, and proposed Agentic Cyber-SLA decision models.",
    label="tab:baseline_comparison"
)

latex_final = final_df.to_latex(
    index=True,
    float_format="%.4f",
    caption="Final performance of the Cyber Risk Agent, SLA Reliability Agent, and Agentic Cyber-SLA Agent.",
    label="tab:final_results"
)

with open(os.path.join(TAB_DIR, "baseline_comparison_latex.tex"), "w") as f:
    f.write(latex_baseline)

with open(os.path.join(TAB_DIR, "final_results_latex.tex"), "w") as f:
    f.write(latex_final)

print("\nSaved LaTeX tables.")

# ============================================================
# 15. Save experiment configuration
# ============================================================

config = {
    "paper_title": "Agentic AI-Driven Cyber-SLA-Aware Edge Intelligence for Secure UAV-Based Aerial Cyber-Physical Systems over 6G Networks",
    "short_title": "Agentic Cyber-SLA Intelligence for Secure UAV-6G Aerial CPS",
    "device": DEVICE,
    "seed": SEED,
    "cyber_dataset_path": cyber_path,
    "sla_dataset_path": sla_path,
    "cyber_shape": list(cyber_df.shape),
    "sla_shape": list(sla_df.shape),
    "cyber_target": cyber_target,
    "sla_target": sla_target,
    "fusion_alpha_cyber": alpha,
    "fusion_beta_sla": beta,
    "decision_actions": [
        "EDGE_OFFLOAD",
        "LOCAL_OR_DELAY",
        "BLOCK_ALERT",
        "SAFE_MODE_ISOLATE"
    ],
    "important_note": (
        "Cyber and SLA datasets are integrated at the decision-framework level, "
        "not as sample-level measurements from the same UAV mission."
    )
}

with open(os.path.join(OUT_DIR, "experiment_config.json"), "w") as f:
    json.dump(config, f, indent=4)

print("\nAll outputs saved in:", OUT_DIR)

# ============================================================
# 16. Print paper-ready interpretation
# ============================================================

print("\n" + "="*80)
print("PAPER-READY INTERPRETATION")
print("="*80)

print("""
This experiment evaluates an Agentic Cyber-SLA framework for secure UAV-based aerial cyber-physical systems over 6G networks.
The Cyber Risk Agent estimates the probability of malicious domain behavior.
The SLA Reliability Agent predicts whether the next 6G telemetry window will satisfy service-level requirements.
The Agentic Decision Agent combines cyber risk and SLA failure risk to select one of four UAV actions:
EDGE_OFFLOAD, LOCAL_OR_DELAY, BLOCK_ALERT, and SAFE_MODE_ISOLATE.

The proposed joint Cyber-SLA agent should be compared against cyber-only and SLA-only baselines.
A strong result is achieved when the joint agent improves ROC-AUC, PR-AUC, F1-score, and unsafe-state detection compared with both single-risk baselines.
""")

print("="*80)

In [ ]:
# ============================================================
# SINGLE CELL: Improve Agentic Cyber-SLA Results
# Weight tuning + threshold tuning + safety metrics + plots
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

# ------------------------------------------------------------
# 1. Check required variables from previous experiment
# ------------------------------------------------------------

required_vars = [
    "cyber_risk_f",
    "sla_failure_risk_f",
    "unsafe_true",
    "OUT_DIR",
    "TAB_DIR",
    "FIG_DIR"
]

missing_vars = [v for v in required_vars if v not in globals()]

if len(missing_vars) > 0:
    raise NameError(
        "Missing required variables from previous notebook code: "
        + ", ".join(missing_vars)
        + "\nRun the original Agentic Cyber-SLA training code first, then run this cell."
    )

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

cyber_risk_f = np.asarray(cyber_risk_f).astype(float)
sla_failure_risk_f = np.asarray(sla_failure_risk_f).astype(float)
unsafe_true = np.asarray(unsafe_true).astype(int)

n = min(len(cyber_risk_f), len(sla_failure_risk_f), len(unsafe_true))

cyber_risk_f = cyber_risk_f[:n]
sla_failure_risk_f = sla_failure_risk_f[:n]
unsafe_true = unsafe_true[:n]

print("Samples used for improved Agentic Cyber-SLA tuning:", n)

# ------------------------------------------------------------
# 2. Evaluation function
# ------------------------------------------------------------

def evaluate_agentic_setting(y_true, scores, threshold):
    y_pred = (scores >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    fpr = fp / (fp + tn + 1e-12)
    fnr = fn / (fn + tp + 1e-12)
    specificity = tn / (tn + fp + 1e-12)
    balanced_accuracy = 0.5 * (recall + specificity)

    try:
        roc_auc = roc_auc_score(y_true, scores)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, scores)
    except Exception:
        pr_auc = np.nan

    return {
        "Threshold": float(threshold),
        "Accuracy": float(accuracy),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "ROC_AUC": float(roc_auc),
        "PR_AUC": float(pr_auc),
        "False_Positive_Rate": float(fpr),
        "False_Negative_Rate": float(fnr),
        "Specificity": float(specificity),
        "Balanced_Accuracy": float(balanced_accuracy),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn)
    }

# ------------------------------------------------------------
# 3. Original result check
# ------------------------------------------------------------

original_alpha = 0.55
original_beta = 0.45
original_threshold = 0.50

original_score = original_alpha * cyber_risk_f + original_beta * sla_failure_risk_f
original_metrics = evaluate_agentic_setting(
    unsafe_true,
    original_score,
    original_threshold
)

print("\nOriginal Agentic Cyber-SLA Result:")
print(pd.DataFrame([original_metrics]).round(4).T)

# ------------------------------------------------------------
# 4. Grid search over cyber/SLA weights and thresholds
# ------------------------------------------------------------

alpha_values = np.arange(0.05, 0.96, 0.05)
threshold_values = np.arange(0.05, 0.96, 0.01)

all_results = []

for alpha in alpha_values:
    beta = 1.0 - alpha
    score = alpha * cyber_risk_f + beta * sla_failure_risk_f

    for threshold in threshold_values:
        result = evaluate_agentic_setting(unsafe_true, score, threshold)
        result["Alpha_Cyber"] = float(alpha)
        result["Beta_SLA"] = float(beta)
        all_results.append(result)

tuning_df = pd.DataFrame(all_results)

tuning_path = os.path.join(TAB_DIR, "agentic_weight_threshold_tuning_results.csv")
tuning_df.to_csv(tuning_path, index=False)

print("\nSaved full tuning search:")
print(tuning_path)
print("Total tested configurations:", len(tuning_df))

# ------------------------------------------------------------
# 5. Best F1 setting
# ------------------------------------------------------------

best_f1_df = tuning_df.sort_values(
    ["F1", "Recall", "Precision", "Accuracy"],
    ascending=False
).head(10)

best_f1_path = os.path.join(TAB_DIR, "best_agentic_f1_settings.csv")
best_f1_df.to_csv(best_f1_path, index=False)

print("\nTop 10 Best F1 Settings:")
print(
    best_f1_df[
        [
            "Alpha_Cyber", "Beta_SLA", "Threshold",
            "Accuracy", "Precision", "Recall", "F1",
            "ROC_AUC", "PR_AUC", "False_Negative_Rate"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 6. Safety setting: recall >= 0.85
# ------------------------------------------------------------

safety_candidates_085 = tuning_df[tuning_df["Recall"] >= 0.85].copy()

if len(safety_candidates_085) > 0:
    best_safety_085_df = safety_candidates_085.sort_values(
        ["F1", "Precision", "Accuracy", "Balanced_Accuracy"],
        ascending=False
    ).head(10)

    safety_085_path = os.path.join(TAB_DIR, "best_agentic_safety_settings_recall_085.csv")
    best_safety_085_df.to_csv(safety_085_path, index=False)

    print("\nTop 10 Safety Settings: Recall >= 0.85")
    print(
        best_safety_085_df[
            [
                "Alpha_Cyber", "Beta_SLA", "Threshold",
                "Accuracy", "Precision", "Recall", "F1",
                "ROC_AUC", "PR_AUC", "False_Negative_Rate"
            ]
        ].round(4)
    )
else:
    print("\nNo setting reached Recall >= 0.85")

# ------------------------------------------------------------
# 7. Strict safety setting: recall >= 0.90
# ------------------------------------------------------------

safety_candidates_090 = tuning_df[tuning_df["Recall"] >= 0.90].copy()

if len(safety_candidates_090) > 0:
    best_safety_090_df = safety_candidates_090.sort_values(
        ["F1", "Precision", "Accuracy", "Balanced_Accuracy"],
        ascending=False
    ).head(10)

    safety_090_path = os.path.join(TAB_DIR, "best_agentic_safety_settings_recall_090.csv")
    best_safety_090_df.to_csv(safety_090_path, index=False)

    print("\nTop 10 Strict Safety Settings: Recall >= 0.90")
    print(
        best_safety_090_df[
            [
                "Alpha_Cyber", "Beta_SLA", "Threshold",
                "Accuracy", "Precision", "Recall", "F1",
                "ROC_AUC", "PR_AUC", "False_Negative_Rate"
            ]
        ].round(4)
    )
else:
    print("\nNo setting reached Recall >= 0.90")

# ------------------------------------------------------------
# 8. Select final improved setting
# Priority:
# 1) Recall >= 0.90 if available
# 2) Recall >= 0.85 if available
# 3) Best F1
# ------------------------------------------------------------

if len(safety_candidates_090) > 0:
    final_row = safety_candidates_090.sort_values(
        ["F1", "Precision", "Accuracy", "Balanced_Accuracy"],
        ascending=False
    ).iloc[0]
    selection_type = "Strict safety: Recall >= 0.90"

elif len(safety_candidates_085) > 0:
    final_row = safety_candidates_085.sort_values(
        ["F1", "Precision", "Accuracy", "Balanced_Accuracy"],
        ascending=False
    ).iloc[0]
    selection_type = "Safety: Recall >= 0.85"

else:
    final_row = tuning_df.sort_values(
        ["F1", "Recall", "Precision", "Accuracy"],
        ascending=False
    ).iloc[0]
    selection_type = "Best F1"

final_alpha = float(final_row["Alpha_Cyber"])
final_beta = float(final_row["Beta_SLA"])
final_threshold = float(final_row["Threshold"])

final_score = final_alpha * cyber_risk_f + final_beta * sla_failure_risk_f
final_pred = (final_score >= final_threshold).astype(int)

final_metrics = evaluate_agentic_setting(
    unsafe_true,
    final_score,
    final_threshold
)

final_metrics["Selection_Type"] = selection_type
final_metrics["Alpha_Cyber"] = final_alpha
final_metrics["Beta_SLA"] = final_beta

final_improved_df = pd.DataFrame([final_metrics])

# Reorder columns
final_improved_df = final_improved_df[
    [
        "Selection_Type", "Alpha_Cyber", "Beta_SLA", "Threshold",
        "Accuracy", "Precision", "Recall", "F1",
        "ROC_AUC", "PR_AUC",
        "False_Positive_Rate", "False_Negative_Rate",
        "Specificity", "Balanced_Accuracy",
        "TP", "TN", "FP", "FN"
    ]
]

final_improved_path = os.path.join(TAB_DIR, "final_improved_agentic_cyber_sla_results.csv")
final_improved_df.to_csv(final_improved_path, index=False)

print("\nFinal Improved Agentic Cyber-SLA Result:")
print(final_improved_df.round(4).T)
print("\nSaved final improved result:")
print(final_improved_path)

# ------------------------------------------------------------
# 9. Compare original vs improved
# ------------------------------------------------------------

comparison_df = pd.DataFrame([
    {
        "Model": "Original Agentic Cyber-SLA",
        "Alpha_Cyber": original_alpha,
        "Beta_SLA": original_beta,
        **original_metrics
    },
    {
        "Model": "Improved Agentic Cyber-SLA",
        "Alpha_Cyber": final_alpha,
        "Beta_SLA": final_beta,
        **final_metrics
    }
])

comparison_df = comparison_df[
    [
        "Model", "Alpha_Cyber", "Beta_SLA", "Threshold",
        "Accuracy", "Precision", "Recall", "F1",
        "ROC_AUC", "PR_AUC",
        "False_Positive_Rate", "False_Negative_Rate",
        "Balanced_Accuracy", "TP", "TN", "FP", "FN"
    ]
]

comparison_path = os.path.join(TAB_DIR, "original_vs_improved_agentic_results.csv")
comparison_df.to_csv(comparison_path, index=False)

print("\nOriginal vs Improved Comparison:")
print(comparison_df.round(4))
print("\nSaved comparison:")
print(comparison_path)

# ------------------------------------------------------------
# 10. Improved Agentic UAV action policy
# ------------------------------------------------------------

def improved_agentic_action(cyber_r, sla_r, combined_risk, threshold):
    cyber_high = cyber_r >= 0.50
    sla_high = sla_r >= 0.50
    joint_high = combined_risk >= threshold

    if not joint_high:
        return "EDGE_OFFLOAD"

    if cyber_high and sla_high:
        return "SAFE_MODE_ISOLATE"
    elif cyber_high and not sla_high:
        return "BLOCK_ALERT"
    elif not cyber_high and sla_high:
        return "LOCAL_OR_DELAY"
    else:
        return "RISK_AWARE_MONITOR"

improved_actions = [
    improved_agentic_action(c, s, r, final_threshold)
    for c, s, r in zip(cyber_risk_f, sla_failure_risk_f, final_score)
]

improved_fusion_df = pd.DataFrame({
    "cyber_risk_prob": cyber_risk_f,
    "sla_failure_risk_prob": sla_failure_risk_f,
    "improved_cyber_sla_risk_score": final_score,
    "unsafe_true": unsafe_true,
    "unsafe_pred": final_pred,
    "improved_agentic_action": improved_actions
})

improved_fusion_path = os.path.join(TAB_DIR, "agentic_cyber_sla_decisions.csv")
improved_fusion_df.to_csv(improved_fusion_path, index=False)

improved_action_summary = (
    improved_fusion_df["agentic_action"]
    .value_counts()
    .reset_index()
)

improved_action_summary.columns = ["Agentic_Action", "Count"]
improved_action_summary["Percentage"] = (
    100 * improved_action_summary["Count"] / improved_action_summary["Count"].sum()
)

improved_action_summary_path = os.path.join(TAB_DIR, "agentic_action_summary.csv")
improved_action_summary.to_csv(improved_action_summary_path, index=False)

print("\nAgentic Action Summary:")
print(improved_action_summary.round(2))
print("\nSaved improved decisions:")
print(improved_fusion_path)
print(improved_action_summary_path)

# ------------------------------------------------------------
# 11. Plots
# ------------------------------------------------------------

def save_confusion_plot(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(5.5, 4.5))
    plt.imshow(cm)
    plt.title(title, fontsize=13)
    plt.xlabel("Predicted State", fontsize=12)
    plt.ylabel("True State", fontsize=12)
    plt.xticks([0, 1], ["Safe", "Unsafe"])
    plt.yticks([0, 1], ["Safe", "Unsafe"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=12)

    plt.colorbar()
    plt.tight_layout()

    path = os.path.join(FIG_DIR, filename)
    plt.savefig(path, dpi=600, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def save_threshold_tradeoff_plot(tuning_df, selected_alpha, selected_threshold):
    subset = tuning_df[np.isclose(tuning_df["Alpha_Cyber"], selected_alpha)].copy()
    subset = subset.sort_values("Threshold")

    plt.figure(figsize=(7, 5))
    plt.plot(subset["Threshold"], subset["Precision"], label="Precision")
    plt.plot(subset["Threshold"], subset["Recall"], label="Recall")
    plt.plot(subset["Threshold"], subset["F1"], label="F1-score")
    plt.axvline(
        selected_threshold,
        linestyle="--",
        label=f"Selected threshold = {selected_threshold:.2f}"
    )

    plt.xlabel("Decision Threshold", fontsize=12)
    plt.ylabel("Score", fontsize=12)
    plt.title("Precision-Recall-F1 Trade-off for Agentic Cyber-SLA Agent", fontsize=13)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    path = os.path.join(FIG_DIR, "threshold_tradeoff_600dpi.png")
    plt.savefig(path, dpi=600, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def save_action_distribution_plot(summary_df):
    plt.figure(figsize=(7, 4))
    plt.bar(summary_df["Agentic_Action"], summary_df["Percentage"])

    plt.ylabel("Percentage (%)", fontsize=12)
    plt.xlabel("Agentic Decision", fontsize=12)
    plt.title("Agentic Cyber-SLA Decision Distribution", fontsize=13)
    plt.xticks(rotation=25, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    path = os.path.join(FIG_DIR, "agentic_action_distribution_600dpi.png")
    plt.savefig(path, dpi=600, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def save_roc_plot(y_true, score, title, filename):
    fpr, tpr, _ = roc_curve(y_true, score)
    auc_value = roc_auc_score(y_true, score)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {auc_value:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.xlabel("False Positive Rate", fontsize=12)
    plt.ylabel("True Positive Rate", fontsize=12)
    plt.title(title, fontsize=13)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    path = os.path.join(FIG_DIR, filename)
    plt.savefig(path, dpi=600, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def save_pr_plot(y_true, score, title, filename):
    precision, recall, _ = precision_recall_curve(y_true, score)
    ap_value = average_precision_score(y_true, score)

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"AP = {ap_value:.4f}")

    plt.xlabel("Recall", fontsize=12)
    plt.ylabel("Precision", fontsize=12)
    plt.title(title, fontsize=13)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    path = os.path.join(FIG_DIR, filename)
    plt.savefig(path, dpi=600, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

save_confusion_plot(
    unsafe_true,
    final_pred,
    "Agentic Cyber-SLA Confusion Matrix",
    "agentic_cyber_sla_confusion_600dpi.png"
)

save_threshold_tradeoff_plot(
    tuning_df,
    final_alpha,
    final_threshold
)

save_action_distribution_plot(
    improved_action_summary
)

save_roc_plot(
    unsafe_true,
    final_score,
    "Agentic Cyber-SLA ROC Curve",
    "agentic_cyber_sla_roc_600dpi.png"
)

save_pr_plot(
    unsafe_true,
    final_score,
    "Agentic Cyber-SLA PR Curve",
    "agentic_cyber_sla_pr_600dpi.png"
)

# ------------------------------------------------------------
# 12. Save LaTeX table
# ------------------------------------------------------------

latex_table = comparison_df.round(4).to_latex(
    index=False,
    caption="Original and improved Agentic Cyber-SLA decision results after Cyber-SLA weight and threshold optimization.",
    label="tab:agentic_results"
)

latex_path = os.path.join(TAB_DIR, "original_vs_improved_agentic_results_latex.tex")

with open(latex_path, "w") as f:
    f.write(latex_table)

print("\nSaved LaTeX table:")
print(latex_path)

# ------------------------------------------------------------
# 13. Final paper-ready interpretation
# ------------------------------------------------------------

orig_recall = original_metrics["Recall"]
orig_f1 = original_metrics["F1"]
orig_fnr = original_metrics["False_Negative_Rate"]

new_recall = final_metrics["Recall"]
new_f1 = final_metrics["F1"]
new_fnr = final_metrics["False_Negative_Rate"]

print("\n" + "=" * 90)
print("FINAL PAPER-READY SUMMARY")
print("=" * 90)

print(f"""
Selected configuration: {selection_type}
Cyber weight alpha: {final_alpha:.2f}
SLA weight beta: {final_beta:.2f}
Decision threshold: {final_threshold:.2f}

Original Agentic Cyber-SLA:
Recall = {orig_recall:.4f}
F1-score = {orig_f1:.4f}
False Negative Rate = {orig_fnr:.4f}

Improved Agentic Cyber-SLA:
Recall = {new_recall:.4f}
F1-score = {new_f1:.4f}
False Negative Rate = {new_fnr:.4f}

Interpretation:
The improved Agentic Cyber-SLA configuration tunes the decision threshold and Cyber-SLA
fusion weights to reduce missed unsafe states. This is important for UAV-based aerial
cyber-physical systems because false negatives represent unsafe communication or cyber
conditions that the system fails to block. The improved configuration should be reported
as the safety-oriented Agentic AI policy.
""")

print("All improved outputs saved in:")
print(OUT_DIR)
print("=" * 90)

In [ ]:
# ============================================================
# ZIP ALL AGENTIC CYBER-SLA RESULTS
# ============================================================

import os
import zipfile
from datetime import datetime

RESULTS_DIR = "/kaggle/working/Agentic_Cyber_SLA_Results"
ZIP_PATH = "/kaggle/working/Agentic_Cyber_SLA_Results.zip"

if not os.path.exists(RESULTS_DIR):
    raise FileNotFoundError(
        f"Results folder not found: {RESULTS_DIR}\n"
        "Run the Agentic Cyber-SLA experiment first."
    )

# Remove old zip if it exists
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

file_count = 0

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(RESULTS_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, RESULTS_DIR)
            zipf.write(file_path, arcname)
            file_count += 1

print("=" * 70)
print("ZIP CREATED SUCCESSFULLY")
print("=" * 70)
print("Source folder:", RESULTS_DIR)
print("Zip file:", ZIP_PATH)
print("Total files zipped:", file_count)
print("Created at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 70)

# Show downloadable file in Kaggle output
print("\nDownload this file from Kaggle output:")
print(ZIP_PATH)